In [1]:
import os
import random
import cv2
import SimpleITK as sitk
import numpy as np

# =========================
# USER INPUT PATHS
# =========================

source_path = r"/mnt/lustre/lalit47/us_data/pre_train/PSFHS/PSFHS"
target_path = r"/mnt/lustre/lalit47/down/seg/PSFHS"

# Source folders
images_folder = os.path.join(source_path, "image_mha")
masks_folder = os.path.join(source_path, "label_mha")

# Target folders
train_images_out = os.path.join(target_path, "train", "images")
train_masks_out = os.path.join(target_path, "train", "masks")

val_images_out = os.path.join(target_path, "val", "images")
val_masks_out = os.path.join(target_path, "val", "masks")

test_images_out = os.path.join(target_path, "test", "images")
test_masks_out = os.path.join(target_path, "test", "masks")

# Create target directories
for folder in [
    train_images_out,
    train_masks_out,
    val_images_out,
    val_masks_out,
    test_images_out,
    test_masks_out,
]:
    os.makedirs(folder, exist_ok=True)


def read_mha_as_2d_array(file_path):
    """
    Read .mha file using SimpleITK and return a 2D numpy array.
    If the file is 3D, the middle slice is used.
    """

    arr = sitk.GetArrayFromImage(sitk.ReadImage(file_path))

    if arr.ndim == 3:
        arr = arr[arr.shape[0] // 2]

    elif arr.ndim == 2:
        pass

    else:
        raise ValueError(f"Unsupported array shape {arr.shape} for file: {file_path}")

    return arr


def normalize_image_to_uint8(image):
    """
    Normalize image array to uint8 range 0-255.
    """

    image = image.astype(np.float32)

    min_val = np.min(image)
    max_val = np.max(image)

    if max_val > min_val:
        image = (image - min_val) / (max_val - min_val)
        image = image * 255.0
    else:
        image = np.zeros_like(image)

    return image.astype(np.uint8)


def normalize_mask_to_uint8(mask):
    """
    Convert mask to uint8 while preserving label values.
    """

    mask = mask.astype(np.uint8)
    return mask


def process_and_save_mha(src, dst, is_mask=False):
    """
    Read .mha, convert to 2D, resize to 256x256, save as PNG.
    """

    try:
        arr = read_mha_as_2d_array(src)
    except Exception as e:
        print(f"Failed to read: {src}")
        print(e)
        return False

    if is_mask:
        arr = normalize_mask_to_uint8(arr)
        arr = cv2.resize(arr, (256, 256), interpolation=cv2.INTER_NEAREST)
    else:
        arr = normalize_image_to_uint8(arr)
        arr = cv2.resize(arr, (256, 256), interpolation=cv2.INTER_LINEAR)

    cv2.imwrite(dst, arr)
    return True


def get_mha_stems(folder):
    """
    Get .mha file names without extension.
    """

    stems = []

    for file in os.listdir(folder):
        if file.lower().endswith(".mha"):
            stem = os.path.splitext(file)[0]
            stems.append(stem)

    stems.sort()
    return stems


def split_train_val_test(stems, train_ratio=0.7, val_ratio=0.1, test_ratio=0.2, seed=42):
    """
    Split data into 7:1:2 train, val, test.
    """

    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, \
        "Ratios must sum to 1"

    random.seed(seed)
    random.shuffle(stems)

    total = len(stems)

    train_end = int(total * train_ratio)
    val_end = train_end + int(total * val_ratio)

    train_stems = stems[:train_end]
    val_stems = stems[train_end:val_end]
    test_stems = stems[val_end:]

    return train_stems, val_stems, test_stems


def process_split(stems, images_out, masks_out):
    """
    Process image-mask pairs for one split.
    Image: image_mha/03744.mha
    Mask: label_mha/03744.mha
    """

    for stem in stems:
        image_src = os.path.join(images_folder, stem + ".mha")
        mask_src = os.path.join(masks_folder, stem + ".mha")

        image_dst = os.path.join(images_out, stem + ".png")
        mask_dst = os.path.join(masks_out, stem + ".png")

        if os.path.exists(image_src):
            process_and_save_mha(image_src, image_dst, is_mask=False)
        else:
            print(f"Image Missing: {image_src}")

        if os.path.exists(mask_src):
            process_and_save_mha(mask_src, mask_dst, is_mask=True)
        else:
            print(f"Mask Missing: {mask_src}")


# =========================
# Main processing
# =========================

image_stems = get_mha_stems(images_folder)
mask_stems = get_mha_stems(masks_folder)

# Use only files that have both image and mask
common_stems = sorted(list(set(image_stems) & set(mask_stems)))

missing_masks = sorted(list(set(image_stems) - set(mask_stems)))
missing_images = sorted(list(set(mask_stems) - set(image_stems)))

if missing_masks:
    print(f"Images without masks: {len(missing_masks)}")

if missing_images:
    print(f"Masks without images: {len(missing_images)}")

train_stems, val_stems, test_stems = split_train_val_test(
    common_stems,
    train_ratio=0.7,
    val_ratio=0.1,
    test_ratio=0.2,
    seed=42
)

print(f"Total paired files: {len(common_stems)}")
print(f"Train files: {len(train_stems)}")
print(f"Val files: {len(val_stems)}")
print(f"Test files: {len(test_stems)}")

process_split(train_stems, train_images_out, train_masks_out)
process_split(val_stems, val_images_out, val_masks_out)
process_split(test_stems, test_images_out, test_masks_out)

print("\nDataset preprocessing completed successfully!")

Total paired files: 1358
Train files: 950
Val files: 135
Test files: 273

Dataset preprocessing completed successfully!
